# Cat Behaviour Video Labeller

Label each cat's behaviour in each video by identifying behaviour transitions.

## Configure notebook

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
import asyncio
import base64
import json
from collections import defaultdict

import cv2
import ipywidgets as widgets
from IPython.display import display
from ipyevents import Event

from tracking import CAT_CLASSIFICATION_MODEL
from utils import get_video_hashes, get_video_paths

In [ ]:
# Define constants
DISPLAY_WIDTH = 960
PLAY_INTERVAL_MS = 140
BEHAVIOURS = [
    "acceptable surface",
    "unacceptable surface",
    "drinking",
    "acceptable bowl",
    "unacceptable bowl",
    "interacting",
]
CAT_NAMES = [str(c) for c in CAT_CLASSIFICATION_MODEL.classes_]

In [ ]:
# Define paths
DATA_ROOT = PROJECT_ROOT / "datasets" / "behaviour_dataset"
ANN_PATH = DATA_ROOT / "behaviour_labels.json"
VIDEO_FILES = get_video_paths(
    mock_inputs=True, mock_inputs_long=True, raw_behaviour=True
)

## Load and validate videos

In [ ]:
class VideoInfo:
    def __init__(self, path: Path):
        self.video_name = path.name
        self.video_path = path
        capture = cv2.VideoCapture(str(path))
        self.frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        capture.release()
        self._images = None

    def __len__(self) -> int:
        return self.frame_count

    @property
    def images(self) -> list:
        if self._images is None:
            capture = cv2.VideoCapture(str(self.video_path))
            self._images = []
            while True:
                success, image = capture.read()
                if not success:
                    break
                self._images.append(image)
            capture.release()
        return self._images

    @property
    def hashes(self) -> tuple[str, ...]:
        return get_video_hashes(self.video_path)

    def close(self) -> None:
        self._images = None


def is_video_annotated(video: VideoInfo, annotations: list[dict]) -> bool:
    # filter annotations for the current video
    video_annotations = [a for a in annotations if a["video_name"] == video.video_name]
    if not video_annotations:
        return False

    # create hash map
    hash_map = {h: i for i, h in enumerate(video.hashes)}

    # convert annotations to index based windows
    windows = defaultdict(list)
    for ann in video_annotations:
        start = hash_map[ann["frame_hash_start"]]
        end = hash_map[ann["frame_hash_end"]]
        windows[ann["cat_id"]].append((start, end))

    # check for overlapping windows
    for cat_id, windows in windows.items():
        windows.sort()
        if any(v[0] <= u[1] for u, v in zip(windows, windows[1:])):
            raise ValueError(f"{video.video_name} annotations overlap for cat {cat_id}")

    return True

In [ ]:
# load annotations and video information
annotations = json.loads(ANN_PATH.read_text()) if ANN_PATH.exists() else []
all_videos = [VideoInfo(video_path) for video_path in VIDEO_FILES]
pending_videos = [v for v in all_videos if not is_video_annotated(v, annotations)]
print(f"Videos for labelling: {len(pending_videos)}")
print(f"Complete videos: {len(all_videos) - len(pending_videos)}")

## Define annotation building and saving

In [ ]:
def build_annotation(
    video: VideoInfo, cat_id: str, start: int, end: int, behaviour: str
) -> dict:
    if not 0 <= start <= end < len(video):
        raise ValueError("Annotation boundaries are out of order")
    return {
        "video_name": video.video_name,
        "cat_id": cat_id,
        "frame_hash_start": video.hashes[start],
        "frame_hash_end": video.hashes[end],
        "behaviour": behaviour,
    }


def save_annotations(rows: list[dict]) -> None:
    rows_to_save = [r for r in rows if r["behaviour"]]
    temp_path = ANN_PATH.with_name(f"{ANN_PATH.name}.tmp")
    temp_path.write_text(json.dumps(rows_to_save, indent=2) + "\n")
    temp_path.replace(ANN_PATH)

## Define interactive annotation player

In [ ]:
def render_video_frame(video: VideoInfo, index: int) -> bytes:
    success, encoded = cv2.imencode(".jpg", video.images[index])
    if not success:
        raise RuntimeError("OpenCV could not encode the preview frame")
    return encoded.tobytes()

In [ ]:
video_index = 0
segment_start = {cat_id: None for cat_id in CAT_NAMES}
active_behaviour = {cat_id: None for cat_id in CAT_NAMES}

frame_view = widgets.HTML(
    layout=widgets.Layout(width=f"{DISPLAY_WIDTH}px", border="1px solid #555")
)
frame_slider = widgets.IntSlider(
    description="Frame",
    min=0,
    max=0,
    value=0,
    continuous_update=True,
    layout=widgets.Layout(width="70%"),
)
play_toggle = widgets.ToggleButton(value=False, description="Play", icon="play")
behaviour_controls = {
    cat_id: widgets.Dropdown(
        options=[
            ("no behaviour", None),
            *((behaviour, behaviour) for behaviour in BEHAVIOURS),
        ],
        value=None,
        description=cat_id,
        layout=widgets.Layout(width="360px"),
    )
    for cat_id in CAT_NAMES
}
start_button = widgets.Button(description="Start behaviour", icon="play")
transition_button = widgets.Button(description="Transition here", icon="exchange")
finish_button = widgets.Button(description="Finish video", icon="check")
keyboard_target = widgets.HTML(
    value=(
        '<div tabindex="0" style="padding:8px;border:1px solid #bbb;max-width:440px">'
        "Click here, then use Space or the arrow keys</div>"
    )
)
keyboard_event = Event(
    source=keyboard_target,
    watched_events=["keydown"],
    prevent_default_action=True,
)
status = widgets.HTML()

In [ ]:
def refresh_view(*_) -> None:
    if not pending_videos or video_index >= len(pending_videos):
        status.value = "All videos have been labelled."
        return

    video = pending_videos[video_index]
    active = ", ".join(
        f"{cat_id}: {active_behaviour[cat_id] or '-'}" for cat_id in CAT_NAMES
    )
    status.value = (
        f"<b>{video.video_name}</b> &nbsp; "
        f"frame {frame_slider.value + 1}/{len(video)}"
        f" &nbsp; hash: {video.hashes[frame_slider.value]}"
        f" &nbsp; active: {active}"
    )
    encoded = base64.b64encode(render_video_frame(video, frame_slider.value)).decode(
        "ascii"
    )
    frame_view.value = (
        f'<img src="data:image/jpeg;base64,{encoded}" '
        'style="display:block;max-width:100%;height:auto">'
    )


def set_frame(index: int) -> None:
    frame_slider.value = max(frame_slider.min, min(index, frame_slider.max))


def current_valid_index() -> int | None:
    if not pending_videos or video_index >= len(pending_videos):
        return None
    return frame_slider.value


def load_next_video() -> None:
    if video_index >= len(pending_videos):
        frame_view.value = ""
        status.value = "All videos have been labelled."
        return

    video = pending_videos[video_index]
    if len(video) == 0:
        frame_view.value = ""
        status.value = f"{video.video_name} contains no decodable frames."
        return

    frame_slider.max = len(video) - 1
    frame_slider.value = 0
    for cat_id in CAT_NAMES:
        segment_start[cat_id] = None
        active_behaviour[cat_id] = None
        behaviour_controls[cat_id].value = None
    play_toggle.value = False
    refresh_view()

In [ ]:
def start_behaviour(*_) -> None:
    start = current_valid_index()
    if start is None:
        status.value = "No frame is available at this position."
        return

    started = []
    for cat_id, control in behaviour_controls.items():
        if active_behaviour[cat_id] is None and control.value is not None:
            segment_start[cat_id] = start
            active_behaviour[cat_id] = control.value
            started.append(cat_id)
    if not started:
        status.value = "Select a behaviour for at least one cat before starting."
        return
    refresh_view()


def save_transition(*_) -> None:
    video = pending_videos[video_index]
    transition = current_valid_index()
    if transition is None:
        status.value = "No frame is available at this position."
        return

    changed = [
        cat_id
        for cat_id, control in behaviour_controls.items()
        if control.value != active_behaviour[cat_id]
    ]
    if not changed:
        status.value = "Select a different behaviour before transitioning."
        return

    if any(
        active_behaviour[cat_id] is not None and transition <= segment_start[cat_id]
        for cat_id in changed
    ):
        status.value = "Move forward to the first frame of the new behaviour."
        return

    for cat_id in changed:
        if active_behaviour[cat_id] is not None:
            annotations.append(
                build_annotation(
                    video,
                    cat_id,
                    segment_start[cat_id],
                    transition - 1,
                    active_behaviour[cat_id],
                )
            )
        if behaviour_controls[cat_id].value is None:
            segment_start[cat_id] = None
            active_behaviour[cat_id] = None
        else:
            segment_start[cat_id] = transition
            active_behaviour[cat_id] = behaviour_controls[cat_id].value

    save_annotations(annotations)
    refresh_view()


def finish_video(*_) -> None:
    global video_index

    if any(
        control.value != active_behaviour[cat_id]
        for cat_id, control in behaviour_controls.items()
    ):
        status.value = "Apply selected behaviour changes with Transition here first."
        return

    video = pending_videos[video_index]
    if len(video) == 0:
        status.value = f"{video.video_name} contains no decodable frames."
        return
    last_index = len(video) - 1
    for cat_id in CAT_NAMES:
        start = segment_start[cat_id]
        behaviour = active_behaviour[cat_id]
        if start is not None and behaviour is not None and start <= last_index:
            annotations.append(
                build_annotation(video, cat_id, start, last_index, behaviour)
            )
    if annotations:
        save_annotations(annotations)
    video.close()
    video_index += 1
    load_next_video()

In [ ]:
def handle_key(event: dict) -> None:
    key = event.get("key", "").lower()
    if key in (" ", "space"):
        play_toggle.value = not play_toggle.value
    elif key in ("arrowleft", "left"):
        set_frame(frame_slider.value - 1)
    elif key in ("arrowright", "right"):
        set_frame(frame_slider.value + 1)


def start_labelling() -> None:
    controls = widgets.VBox(
        [
            widgets.HBox([play_toggle, frame_slider]),
            widgets.VBox(list(behaviour_controls.values())),
            widgets.HBox([start_button, transition_button, finish_button]),
            keyboard_target,
            status,
            frame_view,
        ]
    )
    display(controls)

    keyboard_event.on_dom_event(handle_key)
    frame_slider.observe(refresh_view, names="value")
    for control in behaviour_controls.values():
        control.observe(refresh_view, names="value")
    start_button.on_click(start_behaviour)
    transition_button.on_click(save_transition)
    finish_button.on_click(finish_video)

    async def playback_loop() -> None:
        while play_toggle.value and video_index < len(pending_videos):
            if frame_slider.value >= frame_slider.max:
                play_toggle.value = False
                break
            frame_slider.value += 1
            await asyncio.sleep(PLAY_INTERVAL_MS / 1000)

    def on_play_toggle(change: dict) -> None:
        play_toggle.icon = "pause" if change["new"] else "play"
        if change["new"]:
            asyncio.create_task(playback_loop())

    play_toggle.observe(on_play_toggle, names="value")
    load_next_video()

## Label videos

In [ ]:
start_labelling()